In [1]:
# Another set of controlled experiments, now without the LangChain stuff
# testing a raw ClemAgent playing the clembench games
#credits to Philipp, I just adapted some things a bit

In [2]:
import os

# Specify the game name here (this code can be adapted to any 2-player game)
GAME_NAME = "textmapworld"

# Local clone location of the clembench repository
CLEMBENCH_HOME = r"C:\Users\white\Desktop\agents_experiments\clembench_v3"

# Expose CLEMBENCH_HOME so the clem framework can find the games
os.environ["CLEMBENCH_HOME"] = CLEMBENCH_HOME

In [3]:
# Clone the clembench repo (safe to re-run; git will warn if it already exists)
#!git clone https://github.com/clp-research/clembench $CLEMBENCH_HOME

# Install the requirements into the Python kernel
#%pip install -r $CLEMBENCH_HOME/requirements.txt

# Make tqdm usable in Jupyter notebooks
#%pip install --upgrade ipywidgets jupyter_client

In [4]:
#!pip install clemcore==3.5.0

In [5]:
#!clem --version
#!clem list games -s $GAME_NAME

In [6]:
from playpen.agents import ClemAgent, ClemObservation
from clemcore.backends import load_model


.--------------..--------------..--------------..--------------..--------------..--------------..--------------.
|   ______     ||   _____      ||      __      ||  ____  ____  ||   ______     ||  _________   || ____  _____  |
|  |_   __ \   ||  |_   _|     ||     /  \     || |_  _||_  _| ||  |_   __ \   || |_   ___  |  |||_   \|_   _| |
|    | |__) |  ||    | |       ||    / /\ \    ||   \ \  / /   ||    | |__) |  ||   | |_  \_|  ||  |   \ | |   |
|    |  ___/   ||    | |   _   ||   / ____ \   ||    \ \/ /    ||    |  ___/   ||   |  _|  _   ||  | |\ \| |   |
|   _| |_      ||   _| |__/ |  || _/ /    \ \_ ||    _|  |_    ||   _| |_      ||  _| |___/ |  || _| |_\   |_  |
|  |_____|     ||  |________|  |||____|  |____|||   |______|   ||  |_____|     || |_________|  |||_____|\____| |
'--------------''--------------''--------------''--------------''--------------''--------------''--------------'



## Agent definitions

In [7]:
class BaselineAgentPlayer(ClemAgent):
    """
    Simple example agent.

    It calls the "clp-chat" model with the current interaction history and
    uses the model's response as the next guess.
    """

    def __init__(self):
        super().__init__()
        self.model = load_model("clp-chat", gen_args=dict(temperature=0.0, max_tokens=None))

    def act(self, last: ClemObservation) -> str:
        # Use the full history (which usually already includes the last observation)
        _, _, response_text = self.model.generate_response(self.history)
        # Observe own response in the interaction history
        self.observe(dict(role="user", content=response_text))
        return response_text

In [8]:
class BasePlanningAgent(ClemAgent):
    """
    Simple example agent with an additional planning loop
    """

    def __init__(self):
        super().__init__()
        self.model = load_model("clp-chat", gen_args=dict(temperature=0.0, max_tokens=None))

    def act(self, last: ClemObservation) -> str:
      demand_prompt = self.history + [
          {"role": "user", "content": "You are a skilled planner. Above you see the game history. What would be the best next action? Reply only with \"ACT:\" followed by your action."}
      ]
      _, _, act_response = self.model.generate_response(demand_prompt)

      print("DEBUG: planned response:", act_response)
      if "ACT:" in act_response:
          act_response = act_response.split("ACT:", 1)[1].strip()

      self.observe(dict(role="user", content=act_response))
        
      print("DEBUG: HISTORY")
      print(self.history)
        
      return act_response

In [9]:
class FormatCheckingAgent(ClemAgent):

    """
    Simple example agent with an additional prompt checking loop.
    """
    def __init__(self):
          super().__init__()
          self.model = load_model("clp-chat", gen_args=dict(temperature=0.0, max_tokens=None))
          self.format_requirement = None
          self.max_retries = 3

    def extract_format(self) -> str:
          prompt = self.history + [
              {"role": "user", "content": (
                  "Looking at the instructions above, what is the exact required format for your game responses? "
                  "Extract only the format pattern, e.g. 'CLUE: <text>' or 'GUESS: <word>'. "
                  "Reply with just the format, nothing else."
              )}
          ]
          _, _, fmt = self.model.generate_response(prompt)
          return fmt.strip()

    def is_valid(self, response: str) -> bool:
          """Ask the model whether a response meets the extracted format requirement."""
          prompt = [
              {"role": "user", "content": (
                  f"Required format: {self.format_requirement}\n"
                  f"Response: {response}\n"
                  "Does this response follow the required format? Answer only 'yes' or 'no'."
              )}
          ]
          _, _, verdict = self.model.generate_response(prompt)
          return verdict.strip().lower().startswith("yes")

    def act(self, last: ClemObservation) -> str:
          # Extract format once per game!!
          if self.format_requirement is None:
              self.format_requirement = self.extract_format()

          prompt = self.history.copy()

          for attempt in range(self.max_retries):    #limit the retries attempts
              _, _, response_text = self.model.generate_response(prompt)

              if self.is_valid(response_text):
                  return response_text

              # if not valid, reprompt with format reminder
              prompt = prompt + [
                  {"role": "assistant", "content": response_text},
                  {"role": "user", "content": f"Invalid format. You must follow this format exactly:{self.format_requirement}"} ]

          # return last attempt
          return response_text

In [10]:
#player initialization
player1 = FormatCheckingAgent()
player2 = FormatCheckingAgent()

2026-03-09 17:54:26,255 - clemcore.backends - INFO - Found registered model spec that unifies with {"model_name":"clp-chat"} -> {'model_name': 'clp-chat', 'backend': 'openai_compatible', 'lookup_source': 'C:\\Users\\white\\Desktop\\agents_experiments\\notebooks\\model_registry.json', 'model_id': 'Qwen/Qwen3-VL-30B-A3B-Instruct-FP8'}
2026-03-09 17:54:26,261 - clemcore.backends - INFO - Found registry entry for backend openai_compatible -> {'backend': 'openai_compatible', 'file_name': 'openai_compatible_api.py', 'file_path': 'C:\\Users\\white\\anaconda3\\envs\\playpen-env\\lib\\site-packages\\clemcore\\backends\\openai_compatible_api.py', 'lookup_source': 'packaged'}
2026-03-09 17:54:26,264 - clemcore.backends - INFO - Dynamically import backend openai_compatible
2026-03-09 17:54:27,499 - clemcore.backends - INFO - Successfully loaded clp-chat model
2026-03-09 17:54:27,500 - clemcore.backends - INFO - Loading models took: 0:00:01.237178
2026-03-09 17:54:27,505 - clemcore.backends - INFO 

In [11]:
from clemcore.clemgame import env, episode_results_folder_callbacks

# Create callbacks to record the interactions in a folder; here we name the folder after the models the agent uses
callbacks = episode_results_folder_callbacks(run_dir="clemagent", result_dir_path="playpen-records", player_model_infos="FormatCheckingAgent")

game_env = env(
    GAME_NAME,
    single_pass=True,
    callbacks=callbacks
)

game_env.reset()

2026-03-09 17:54:27,601 - clemcore.cli - INFO - Found '1' game matching the game_selector="textmapworld"
2026-03-09 17:54:27,603 - clemcore.cli - INFO - {
  "game_name": "textmapworld",
  "description": "Exhaustively exploring a map.",
  "main_game": "textmapworld",
  "players": 1,
  "image": "none",
  "languages": [
    "en"
  ],
  "benchmark": [
    "2.0",
    "3.0"
  ],
  "regression": "large",
  "roles": [
    "Describer"
  ],
  "game_path": "C:\\Users\\white\\Desktop\\agents_experiments\\clembench_v3\\textmapworld\\textmapworld_main"
}
2026-03-09 17:54:27,605 - clemcore.run - INFO - Loading game benchmark for textmapworld
2026-03-09 17:54:28,510 - clemcore.run - INFO - Loading game benchmark for textmapworld took: 0:00:00.902002
2026-03-09 17:54:28,515 - clemcore.run - INFO - Prepared instance queue for textmapworld using 5 experiments ['small', 'medium', 'large', 'medium_cycle', 'large_cycle'] and 50 instances in total.
2026-03-09 17:54:28,518 - clemcore.run - INFO - Detected sin

In [12]:
print("possible agents:", game_env.possible_agents)
# In most cases, roles will be the content description of what player_0 and player_1 are
print("likely mapping:", game_env.unwrapped.game_master.game_spec["roles"])
agent_mapping = {"player_0": player1, "player_1": game_env.unwrapped.game_master.describer}

possible agents: ['player_0', 'player_1']
likely mapping: ['Describer']


In [13]:
num_episodes = 50

all_episodes_data = []

for episode in range(num_episodes):
    game_env.reset()
    player1.reset()  #keep in mind that if guesser is an agent, then IT should be reset
    player2.reset()

    context_response_pairs = []
    for agent_id in game_env.agent_iter():
        context, reward, termination, truncation, info = game_env.last()
        if termination or truncation:
            response = None # we step one more time to remove the agent from the env (final reward was observed in last)
        else:
            response = agent_mapping[agent_id](context)
        context_response_pairs.append((agent_id, context, response, reward))
        game_env.step(response)

    all_episodes_data.append(context_response_pairs)

    print(f"Episode {episode + 1}/{num_episodes} completed with {len(context_response_pairs)} steps")

    print(f"Episode took these {len(context_response_pairs)} steps:")
    print("-" * 20)
    for idx, (agent_id, context, response, reward) in enumerate(context_response_pairs):
        print(f"Step {idx} / Reward {reward:.2f}:")
        print(f"Agent({agent_id}) <- Context:", context)
        print(f"Agent({agent_id}) -> Response:", response)
        print("-" * 20)

Episode 1/50 completed with 9 steps
Episode took these 9 steps:
--------------------
Step 0 / Reward 0.00:
Agent(player_0) <- Context: {'role': 'user', 'content': 'Please help me with the following task. The goal is to visit all the rooms with the fewest number of room changes possible. In each room, you need to decide the direction to go in. Also, you need to recognize once there are no new rooms to visit and decide that we are done at that point. Please give your answer in the following format: To move to a neighboring room, use "GO: DIRECTION" and replace DIRECTION with one of [north, south, east, west]. To stop the exploration, answer with "DONE" instead. Omit any other text.\nHere is an example:\nYou are in the Kitchen. Currently available directions: south, west. What is your next instruction?\nGO: west\nYou have made a step and entered a Lobby. Currently available directions: east, north. What is your next instruction?\nGO: north\n...\nYou have made a step and entered a Bedroom.

StopIteration: 